In [ ]:
# @title Set ONLINE = False if running locally
ONLINE = True

In [ ]:

!pip install myfm

import pandas as pd
import numpy as np
import math
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from scipy import sparse as sps
from collections import defaultdict
import myfm
import lightgbm as lgb
import gdown
import os

In [ ]:
# Download training and submission data
if ONLINE:
    url = 'https://drive.google.com/file/d/1lDEtOq644qlaw_vh0i5c3ChVfrHJiuMc/view?usp=sharing'
    output = 'data_train.csv'
    gdown.download(url, output, fuzzy=True)
    url = 'https://drive.google.com/file/d/1KMiakfzTWYr4QJU9S5DC9e5KIo_8bb24/view?usp=sharing'
    output = 'sampleSubmission.csv'
    gdown.download(url, output, fuzzy=True)

In [ ]:
# Define constants
NUMBER_OF_USERS = 10000
NUMBER_OF_MOVIES = 1000
N_FACTORS = 48
N_ITER = 500
RANDOM_SEED = 42
TRAIN_SIZE = 0.9
PREDICTIONS_PATH = 'predictions'
CSV_PATH = 'csvs'
MIN_RATING = 1.0
MAX_RATING = 5.0

# Load data
data_pd = pd.read_csv('data_train.csv')
sample_submission = pd.read_csv('sampleSubmission.csv')

# Extract user and item IDs globally for submission
user_ids = sample_submission['Id'].apply(lambda x: int(x.split('_')[0][1:]) - 1).values
item_ids = sample_submission['Id'].apply(lambda x: int(x.split('_')[1][1:]) - 1).values

def extract_users_items_predictions(data):
    """
    Extract users, items (movies), and predictions from the dataset.
    """
    users, movies = data.Id.str.extract('r(\d+)_c(\d+)').astype(int).values.T - 1
    predictions = data.Prediction.values
    return users, movies, predictions

def prepare_data(data_pd, train_size=TRAIN_SIZE, random_seed=RANDOM_SEED):
    """
    Prepare train and test datasets.
    """
    train_pd, test_pd = train_test_split(data_pd, train_size=train_size, random_state=random_seed)
    train_users, train_movies, train_predictions = extract_users_items_predictions(train_pd)
    test_users, test_movies, test_predictions = extract_users_items_predictions(test_pd)
    return train_users, train_movies, train_predictions, test_users, test_movies, test_predictions

def create_masked_data(train_users, train_movies, train_predictions, n_users, n_movies):
    """
    Create full matrix of observed values.
    """
    data = np.full((n_users, n_movies), np.mean(train_predictions))
    mask = np.zeros((n_users, n_movies))  # 0 -> unobserved value, 1 -> observed value

    for user, movie, pred in zip(train_users, train_movies, train_predictions):
        data[user, movie] = pred
        mask[user, movie] = 1
    return data, mask

def rmse(x, y):
    """
    Calculate Root Mean Squared Error (RMSE).
    """
    return math.sqrt(mean_squared_error(x, y))

class BFM:
    def __init__(self, n_users, n_items, n_factors, n_iter, random_state=None, with_iu=False, with_ii=False, with_ord=False):
        """
        Initialize the Bayesian Factorization Machine (BFM) model.
        """
        self.n_users = n_users
        self.n_items = n_items
        self.n_factors = n_factors
        self.n_iter = n_iter
        self.with_iu = with_iu
        self.with_ii = with_ii
        self.with_ord = with_ord
        if random_state is not None:
            np.random.seed(random_state)

    def fit(self, X, y):
        """
        Fit the BFM model to the training data.
        """
        # Extract unique user and item indices
        train_uid_unique, train_uid_index = np.unique(X[:, 0], return_inverse=True)
        train_mid_unique, train_mid_index = np.unique(X[:, 1], return_inverse=True)

        # Augment user and item data with implicit features if required
        user_data_train = self._augment_user_id(train_uid_unique)
        movie_data_train = self._augment_movie_id(train_mid_unique)

        # Create relational blocks for users and items
        block_user_train = myfm.RelationBlock(train_uid_index, user_data_train)
        block_movie_train = myfm.RelationBlock(train_mid_index, movie_data_train)

        # Initialize the model based on whether Ordered Probit is used
        if self.with_ord:
            self.model = myfm.MyFMOrderedProbit(rank=self.n_factors, random_seed=RANDOM_SEED)
            y = y - 1  # Shift ratings from 1-5 to 0-4 for Ordered Probit
        else:
            self.model = myfm.MyFMRegressor(rank=self.n_factors, random_seed=RANDOM_SEED)

        # Fit the model to the data
        self.model.fit(None, y, n_iter=self.n_iter, X_rel=[block_user_train, block_movie_train])

    def predict(self, users, movies):
        """
        Predict ratings for the given user and item indices.
        """
        # Extract unique user and item indices
        test_uid_unique, test_uid_index = np.unique(users, return_inverse=True)
        test_mid_unique, test_mid_index = np.unique(movies, return_inverse=True)

        # Augment user and item data with implicit features if required
        user_data_test = self._augment_user_id(test_uid_unique)
        movie_data_test = self._augment_movie_id(test_mid_unique)

        # Create relational blocks for users and items
        block_user_test = myfm.RelationBlock(test_uid_index, user_data_test)
        block_movie_test = myfm.RelationBlock(test_mid_index, movie_data_test)
        X_test = [block_user_test, block_movie_test]

        # Predict ratings
        if self.with_ord:
            ordinal_probs = self.model.predict_proba(None, X_test)
            predictions = ordinal_probs.dot(np.arange(1, 6))  # Convert back to 1-5 scale
        else:
            predictions = self.model.predict(None, X_test)

        return predictions

    def _augment_user_id(self, user_ids):
        """
        Augment user data with implicit features if required.
        """
        Xs = [sps.lil_matrix((len(user_ids), self.n_users))]
        for index, user_id in enumerate(user_ids):
            Xs[0][index, user_id] = 1
        if self.with_iu:
            X_iu = sps.lil_matrix((len(user_ids), self.n_items))
            for index, user_id in enumerate(user_ids):
                watched_movies = self.user_vs_watched.get(user_id, [])
                normalizer = 1 / max(len(watched_movies), 1) ** 0.5
                for mid in watched_movies:
                    X_iu[index, mid] = normalizer
            Xs.append(X_iu)
        return sps.hstack(Xs, format='csr')

    def _augment_movie_id(self, movie_ids):
        """
        Augment movie data with implicit features if required.
        """
        Xs = [sps.lil_matrix((len(movie_ids), self.n_items))]
        for index, movie_id in enumerate(movie_ids):
            Xs[0][index, movie_id] = 1
        if self.with_ii:
            X_ii = sps.lil_matrix((len(movie_ids), self.n_users))
            for index, movie_id in enumerate(movie_ids):
                watched_users = self.movie_vs_watched.get(movie_id, [])
                normalizer = 1 / max(len(watched_users), 1) ** 0.5
                for uid in watched_users:
                    X_ii[index, uid] = normalizer
            Xs.append(X_ii)
        return sps.hstack(Xs, format='csr')

    def setup_implicit_data(self, X):
        """
        Set up implicit data for users and movies.
        """
        self.movie_vs_watched = defaultdict(list)
        self.user_vs_watched = defaultdict(list)
        for row in X:
            user_id, movie_id = row[0], row[1]
            self.movie_vs_watched[movie_id].append(user_id)
            self.user_vs_watched[user_id].append(movie_id)

def train_bfm(X_train, y_train, n_factors=N_FACTORS, n_iter=N_ITER, with_ord=False):
    """
    Train Bayesian Factorization Machine model.
    """
    bfm_model = BFM(n_users=NUMBER_OF_USERS, n_items=NUMBER_OF_MOVIES, n_factors=n_factors, n_iter=n_iter, random_state=RANDOM_SEED, with_iu=True, with_ii=True, with_ord=with_ord)
    bfm_model.setup_implicit_data(X_train)
    bfm_model.fit(X_train, y_train)
    return bfm_model

def train_lightgbm(train_users, train_movies, train_residuals):
    """
    Train LightGBM model on residuals.
    """
    # Prepare data for LightGBM
    X_train_lgb = np.column_stack((train_users, train_movies))
    y_train_lgb = train_residuals

    # Create LightGBM dataset
    lgb_train_data = lgb.Dataset(X_train_lgb, label=y_train_lgb)

    # Set LightGBM parameters
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.25,
        'num_leaves': 6,
        'feature_fraction': 0.9,
        'verbose' : -1
    }

    # Train LightGBM model on residuals
    lgb_model = lgb.train(params, lgb_train_data, num_boost_round=100)

    return lgb_model

def evaluate_model(bfm_model, lgb_model, test_users, test_movies, test_predictions):
    """
    Evaluate the combined BFM and LightGBM model.
    """
    # Predict using BFM model
    bfm_predictions = np.clip(bfm_model.predict(test_users, test_movies),MIN_RATING,MAX_RATING)

    # Prepare test data for LightGBM
    X_test_lgb = np.column_stack((test_users, test_movies))

    # Predict residuals using LightGBM model
    lgb_residuals = lgb_model.predict(X_test_lgb)

    # Combine BFM predictions and LightGBM residuals
    combined_test_predictions = bfm_predictions + lgb_residuals

    # Clip combined predictions to ensure they are within the range [1, 5]
    combined_test_predictions_clipped = np.clip(combined_test_predictions, 1, 5)

    # Calculate RMSE
    final_rmse = rmse(combined_test_predictions_clipped, test_predictions)
    return final_rmse, combined_test_predictions_clipped

def save_predictions(predictions, submission_predictions, file_name):
    """
    Save both test and submission predictions in a single .npz file.
    """
    if not os.path.exists(PREDICTIONS_PATH):
        os.mkdir(PREDICTIONS_PATH)

    np.savez_compressed(
        os.path.join(PREDICTIONS_PATH, file_name),
        test=predictions,
        challenge=submission_predictions
    )

def save_submission(submission_predictions, file_name):
    """
    Save submission predictions to CSV file.
    """
    sample_submission['Prediction'] = submission_predictions

    if not os.path.exists(CSV_PATH):
        os.mkdir(CSV_PATH)

    sample_submission[['Id', 'Prediction']].to_csv(os.path.join(CSV_PATH, file_name), index=False)

# Prepare train and test data
train_users, train_movies, train_predictions, test_users, test_movies, test_predictions = prepare_data(data_pd)

# Create full matrix of observed values
data, mask = create_masked_data(train_users, train_movies, train_predictions, NUMBER_OF_USERS, NUMBER_OF_MOVIES)

In [ ]:
# Train BFM Regression Model
bfm_model_reg = train_bfm(np.column_stack((train_users, train_movies)), train_predictions)

# Calculate residuals on train set
bfm_reg_train_predictions = bfm_model_reg.predict(train_users, train_movies)
train_residuals = train_predictions - bfm_reg_train_predictions

# Train LightGBM on residuals
lgb_model_reg = train_lightgbm(train_users, train_movies, train_residuals)

# Evaluate Regression Model
final_rmse_reg, combined_reg_test_predictions = evaluate_model(bfm_model_reg, lgb_model_reg, test_users, test_movies, test_predictions)
print(f'Final RMSE with LightGBM and BFM Regression: {final_rmse_reg}')

# Predict submission for Regression Model
bfm_submission_predictions_reg = bfm_model_reg.predict(user_ids, item_ids)
X_lgb_submission = np.column_stack((user_ids, item_ids))
lgb_submission_residuals_reg = lgb_model_reg.predict(X_lgb_submission)
submission_predictions_reg = bfm_submission_predictions_reg + lgb_submission_residuals_reg

# Clip submission predictions to ensure they are within the range [1, 5]
submission_predictions_reg_clipped = np.clip(submission_predictions_reg, MIN_RATING, MAX_RATING)

# Save Regression Model Predictions
save_predictions(combined_reg_test_predictions, submission_predictions_reg_clipped, 'bmf_reg_predictions.npz')

# Save submission for Regression Model
save_submission(submission_predictions_reg_clipped, 'submission_bmf_reg.csv')

print("Submission file saved successfully.")

In [ ]:
# Train BFM Ordered Probit Model
bfm_model_ord = train_bfm(np.column_stack((train_users, train_movies)), train_predictions, with_ord=True)

# Calculate residuals on train set
bfm_ord_train_predictions = bfm_model_ord.predict(train_users, train_movies)
train_residuals_ord = train_predictions - bfm_ord_train_predictions

# Train LightGBM on residuals
lgb_model_ord = train_lightgbm(train_users, train_movies, train_residuals_ord)

# Evaluate Ordered Probit Model
final_rmse_ord, combined_ord_test_predictions = evaluate_model(bfm_model_ord, lgb_model_ord, test_users, test_movies, test_predictions)
print(f'Final RMSE with LightGBM and BFM Ordered Probit: {final_rmse_ord}')

# Predict submission for Ordered Probit Model
bfm_submission_predictions_ord = bfm_model_ord.predict(user_ids, item_ids)
lgb_submission_residuals_ord = lgb_model_ord.predict(X_lgb_submission)
submission_predictions_ord = bfm_submission_predictions_ord + lgb_submission_residuals_ord

# Clip submission predictions to ensure they are within the range [1, 5]
submission_predictions_ord_clipped = np.clip(submission_predictions_ord, MIN_RATING,MAX_RATING)

# Save Ordered Probit Model Predictions
save_predictions(combined_ord_test_predictions, submission_predictions_ord_clipped, 'bmf_ord_predictions.npz')

# Save submission for Ordered Probit Model
save_submission(submission_predictions_ord_clipped, 'submission_bmf_ord.csv')

print("Submission file saved successfully.")